# Install Nebula

To run nebula scirbt you need the follwing libraries:

```R
    library(glue)
    library(dplyr)
    library(getopt)
    library(Matrix)
    library(qs)
    library(Seurat)
    library(nebula)
    library(future)
```

```R
install.packages("getopt")

if (!require("devtools")) install.packages("devtools")
devtools::install_github("lhe17/nebula")

if (!require("BiocManager")) install.packages("BiocManager")
BiocManager::install("biomaRt")

```

R scritp from [link](https://github.com/bdferris642/sc-online/tree/main/scripts)

# Hyperparameters

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from scipy import sparse
import scanpy as sc
import matplotlib.pyplot as plt
import os
import rpy2.robjects as ro

from dotenv import load_dotenv; load_dotenv()


from utils import preprocessing
from utils import DEG

# ATTENTION: need for perpty of running jupternotbook
%matplotlib inline

PARALLEL_NEBULA_SCRIPT_PATH = os.getenv("PARALLEL_NEBULA_SCRIPT")

DATASET_FOLDER = "/home/gdallagl/myworkdir/XDP/data/XDP/artificial_bican/geneset_001/depleted_versions"
TRUE_DEG_PATH = f"{DATASET_FOLDER}/../genes.csv"
QS_TMP_FOLDER = "/home/gdallagl/myworkdir/XDP/data/_old/nebula_tmp" # tmp folder where to save intemrediate results
DESEQ2_NAME = "baseline_deseq2"
EDGER_NAME = "baseline_edgeR"

# name of cell tyoe vauble annotaton to use in this analsys
CT_FOR_DEG_VARIABLE = "spn_type" #"Group_name",  "spn_type"
# name of Sample varibale
SAMPLE_VARIABLE = "donor_id"
# variable to test if differtially epxressed
CONTRAST_VARIABLE = "case_control"
# Level of contrat varibale to use as baseline
CONTRAST_BASELINE = "control"
# Level of contrat varibale to use as stimulated
CONTRAST_STIM = "case"

# Covariates to use for stat test
COVARIATES_FOR_DEG = ["Age.at.Death", "Sex", "PMI", "pct_mt", "pct_intronic",
                      ]
# libraru size col for  enbula
LIBRARY_SIZE_COL = "nCount_RNA"

# mitluple etst correction pvalue thr
ALPHA_MULTIPLE_TEST = 0.05
LOGFC_THR=0.1
PVAL_THR=0.05

# EdgeR Desseq hyperparaters

# mitluple etst correction pvalue thr
ALPHA_MULTIPLE_TEST = 0.05
# Filter varibales for pseudobulk cells
MIN_CELLS_PER_PSUDOCELL=10
MIN_COUNTS_PER_PSEUDOCELL=1000
MIN_PSEUDOCELL_PER_GROUP=2
# Filter gens noisy/low expressed
MIN_COUNTS=10
LARGE_N=10
MIN_TOTAL_COUNTS=15
MIN_PROP_BY_EXPR=0.5
MIN_PROP_BY_PROP=0.1
MIN_SMPLS=2
#columns names
P_VAL_COL_NAME = f"p_{CONTRAST_VARIABLE}{CONTRAST_STIM}"
ADJ_P_VAL_COL_NAME = f"adj_{P_VAL_COL_NAME}"
LOGFC_COL_NAME = f"logFC_{CONTRAST_VARIABLE}{CONTRAST_STIM}"
GENE_COL_NAME = "gene"
N_TOP_GENES_TO_NAME = 10
# thr for gene signifcat
LOGFC_THR=0.1

FULL_MODEL = "~ " + " + ".join([CONTRAST_VARIABLE, *COVARIATES_FOR_DEG])
print(FULL_MODEL)

~ case_control + Age.at.Death + Sex + PMI + pct_mt + pct_intronic


# Create Covs and update .qs File

Each method will add its covaritates in .qs obj (needed for Nebula).

In [3]:
for dataset in sorted(os.listdir(DATASET_FOLDER)):

    print(f"\n######\n Processing dataset: {dataset} \n######\n")

    # 2 differt saving folders
    results_folder_deseq2 = os.path.join(DATASET_FOLDER, dataset, DESEQ2_NAME)
    os.makedirs(results_folder_deseq2, exist_ok=True)
    results_folder_edgeR = os.path.join(DATASET_FOLDER, dataset, EDGER_NAME)
    os.makedirs(results_folder_edgeR, exist_ok=True)

    # same  initial adata
    dataset_path_h5ad = os.path.join(DATASET_FOLDER, dataset, f"{dataset}_pseudobulked.h5ad")
    cov_df_csv_path = os.path.join(results_folder_deseq2, "covariates_df.csv") # ny defult save in desseq

    ###################

    # Logic to calcualte specific Covs --> df with covariates (cov_df_csv_path)


    ##################

    # Update obs of .qs obj
    # add_metadata_to_qs(
    #     base_qs_path=dataset_path_qs,
    #     new_metadata_df=pd.read_csv(cov_df_csv_path),
    #     barcode_col="barcode",
    # )

    #break


######
 Processing dataset: depleted_0.00_dorsal_matrix 
######


######
 Processing dataset: depleted_0.01_dorsal_matrix 
######


######
 Processing dataset: depleted_0.05_dorsal_matrix 
######


######
 Processing dataset: depleted_0.10_dorsal_matrix 
######


######
 Processing dataset: depleted_0.20_dorsal_matrix 
######


######
 Processing dataset: depleted_0.30_dorsal_matrix 
######


######
 Processing dataset: depleted_0.40_dorsal_matrix 
######


######
 Processing dataset: depleted_0.50_dorsal_matrix 
######



# Psudobulk DEG

In [3]:
for dataset in sorted(os.listdir(DATASET_FOLDER)):

    print(f"\n######\n Processing dataset: {dataset} \n######\n")

    # 2 differt saving folders
    results_folder_deseq2 = os.path.join(DATASET_FOLDER, dataset, DESEQ2_NAME)
    save_csv_path_deseq2 = os.path.join(results_folder_deseq2, f"{dataset}-{DESEQ2_NAME}_results.csv")
    results_folder_edgeR = os.path.join(DATASET_FOLDER, dataset, EDGER_NAME)
    save_csv_path_edgeR = os.path.join(results_folder_edgeR, f"{dataset}-{EDGER_NAME}_results.csv")

    # same  initial adata
    dataset_path_h5ad = os.path.join(DATASET_FOLDER, dataset, f"{dataset}_pseudobulked.h5ad")
    results_folder =results_folder_deseq2 # ny defult save in desseq


    # Read file
    adata_psuedo = sc.read_h5ad(dataset_path_h5ad)

    # Check for collinearity
    #DEG.check_corr_cov_in_design(adata_psuedo, FULL_MODEL, corr_thr=0.7, split=" + ")

    # Run psuedobulk
    try:
        original_show = plt.show

        df_merged, res_df_edgeR, res_df_pds2 = DEG.DEG_deseq2_edgeR(
            adata_pb_all=adata_psuedo,
            psuedobulk_group_for_deg=None, # do not subset
            psuedobulk_group_for_deg_col=None, # do not subset
            design_formula=FULL_MODEL,
            save_folder=results_folder,
            SAMPLE_VARIABLE=SAMPLE_VARIABLE,
            CONTRAST_VARIABLE=CONTRAST_VARIABLE,
            MIN_CELLS_PER_PSUDOCELL=MIN_CELLS_PER_PSUDOCELL,
            MIN_COUNTS_PER_PSEUDOCELL=MIN_COUNTS_PER_PSEUDOCELL,
            MIN_PSEUDOCELL_PER_GROUP=MIN_PSEUDOCELL_PER_GROUP,
            MIN_COUNTS=MIN_COUNTS,
            LARGE_N=LARGE_N,
            MIN_TOTAL_COUNTS=MIN_TOTAL_COUNTS,
            MIN_PROP_BY_EXPR=MIN_PROP_BY_EXPR,
            MIN_PROP_BY_PROP=MIN_PROP_BY_PROP,
            MIN_SMPLS=MIN_SMPLS,
            CONTRAST_BASELINE=CONTRAST_BASELINE,
            CONTRAST_STIM=CONTRAST_STIM,
            ALPHA_MULTIPLE_TEST=ALPHA_MULTIPLE_TEST,
            LOGFC_THR=LOGFC_THR,
            PVAL_THR=PVAL_THR,
            N_TOP_GENES_TO_NAME=N_TOP_GENES_TO_NAME,
            GENE_COL_NAME=GENE_COL_NAME,
            LOGFC_COL_NAME=LOGFC_COL_NAME,
            ADJ_P_VAL_COL_NAME=ADJ_P_VAL_COL_NAME,
            P_VAL_COL_NAME=P_VAL_COL_NAME, # Rename p.val col
            calculate_umap=False
        )

        res_df_edgeR.to_csv(save_csv_path_edgeR, index=False)
        res_df_pds2.to_csv(save_csv_path_deseq2, index=False)


    finally:
        plt.show = original_show

    #break

    


######
 Processing dataset: depleted_0.00_dorsal_matrix 
######


            #########################
            ### Processing psudobulk group: None
            #########################
            

Corrected dsign Matrix: ~ case_control + Age.at.Death + Sex + PMI + pct_mt + pct_intronic
~ case_control + Age.at.Death + Sex + PMI + pct_mt + pct_intronic
['case_control', 'Age.at.Death', 'Sex', 'PMI', 'pct_mt', 'pct_intronic']

 High correlations (|r| > 0.7):
Samples per group: {'case': np.int64(10), 'control': np.int64(9)}

Design formula: ~ case_control + Age.at.Death + Sex + PMI + pct_mt + pct_intronic



/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "pct_ribosomal". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "Neighborhood_bootstrapping_probability". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
2026-01-31 16:22:13 | [WARNING] R callback write-console: In addition:   
2026-01-31 16:22:13 | [WARNING] R callback write-console: Warning messages:
  
2026-01-31 16:22:13 | [WARNING] R callback write-console: 1:   
2026-01-31 16:22:13 | [WARNING] R callback write

Using None as control genes, passed at DeseqDataSet initialization


/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/pydeseq2/utils.py:232: RuntimeWarning: invalid value encountered in multiply
  - counts * np.log(mu)
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/pydeseq2/utils.py:232: RuntimeWarning: invalid value encountered in multiply
  - counts * np.log(mu)
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/pydeseq2/utils.py:385: RuntimeWarning: overflow encountered in multiply
  + ((1 / disp + counts) * mu_ / (1 / disp + mu_)) @ X
Fitting dispersions...
... done in 2.04 seconds.

Fitting dispersion trend curve...
... done in 0.50 seconds.

Fitting MAP dispersions...
... done in 2.26 seconds.

Fitting LFCs...
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/pydeseq2/utils.py:232: RuntimeWarning: invalid value encountered in multiply
  - counts * np.log(mu)
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/pydeseq2/utils.py:385: RuntimeWarning: overflow encountered in multip

Log2 fold change & Wald test p-value, contrast vector: [ 0. -1.  0.  0.  0.  0.  0.]
              baseMean  log2FoldChange     lfcSE      stat    pvalue     padj
A1BG       1345.133735       -0.156603  0.096846 -1.617029  0.105872  0.99946
A1BG-AS1   4047.997202       -0.074461  0.105612 -0.705049  0.480779  0.99946
A2M         436.799364        0.464638  0.303575  1.530554  0.125880  0.99946
A2M-AS1     706.475138        0.069181  0.153578  0.450463  0.652377  0.99946
A2ML1       532.035328       -0.085270  0.157971 -0.539782  0.589347  0.99946
...                ...             ...       ...       ...       ...      ...
ZXDC      21821.105539       -0.026604  0.083719 -0.317775  0.750655  0.99946
ZYG11B    16862.373348        0.018369  0.062656  0.293171  0.769391  0.99946
ZYX        3772.659597        0.013418  0.132767  0.101067  0.919497  0.99946
ZZEF1     26552.335055       -0.047144  0.055031 -0.856682  0.391621  0.99946
ZZZ3      47774.262175        0.174252  0.076066  2.29081

...

Log2 fold change & Wald test p-value, contrast vector: [ 0. -1.  0.  0.  0.  0.  0.]
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG       1082.332267       -0.076421  0.102542 -0.745264  0.456112  0.932264
A1BG-AS1   3127.811729       -0.101307  0.114854 -0.882046  0.377752  0.913426
A2M         364.949922        0.702512  0.315586  2.226056  0.026010  0.501764
A2M-AS1     555.401622        0.134942  0.162109  0.832417  0.405173  0.917155
A2ML1       412.593254       -0.139455  0.160252 -0.870225  0.384177  0.914667
...                ...             ...       ...       ...       ...       ...
ZXDC      16888.038427       -0.022089  0.089865 -0.245798  0.805839  0.979833
ZYG11B    13168.110121        0.018526  0.070195  0.263918  0.791843  0.976749
ZYX        2985.217285        0.021780  0.147700  0.147464  0.882765  0.987714
ZZEF1     20466.235769       -0.062369  0.060360 -1.033268  0.301479  0.888515
ZZZ3      37583.881611        0.249538  0.0835

/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "pct_ribosomal". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "Neighborhood_bootstrapping_probability". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
2026-01-31 16:25:56 | [WARNING] R callback write-console: In addition:   
2026-01-31 16:25:56 | [WARNING] R callback write-console: Warning messages:
  
2026-01-31 16:25:56 | [WARNING] R callback write-console: 1:   
2026-01-31 16:25:56 | [WARNING] R callback write

Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.88 seconds.

Fitting dispersion trend curve...
... done in 0.50 seconds.

Fitting MAP dispersions...
... done in 2.23 seconds.

Fitting LFCs...
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/pydeseq2/utils.py:232: RuntimeWarning: invalid value encountered in multiply
  - counts * np.log(mu)
... done in 2.26 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 1.19 seconds.



Log2 fold change & Wald test p-value, contrast vector: [ 0. -1.  0.  0.  0.  0.  0.]
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG       1004.705576       -0.028969  0.107688 -0.269008  0.787924  0.958192
A1BG-AS1   2850.887363       -0.093554  0.116684 -0.801767  0.422688  0.854334
A2M         339.373462        0.757415  0.326406  2.320469  0.020315  0.310288
A2M-AS1     509.535962        0.181709  0.166009  1.094574  0.273703  0.771158
A2ML1       375.622562       -0.170555  0.166031 -1.027248  0.304303  0.790348
...                ...             ...       ...       ...       ...       ...
ZXDC      15347.877218       -0.024076  0.091027 -0.264496  0.791398  0.958474
ZYG11B    12033.073255        0.016371  0.072224  0.226665  0.820684  0.963563
ZYX        2754.789639        0.047675  0.154868  0.307840  0.758204  0.950636
ZZEF1     18620.090884       -0.062333  0.062057 -1.004442  0.315166  0.798376
ZZZ3      34286.944906        0.265999  0.0860

In [21]:
res_df_pds2

,gene,baseMean,logFC_case_controlcase,lfcSE,stat,p_case_controlcase,adj_p_case_controlcase,contrast,method,is_significant
20,LINC03033,1635.862208,-2.539983,0.438527,-5.792076,6.952166e-09,5.636882e-06,None,Deseq2,True
29,SPIRE2,7592.893923,-2.201442,0.428043,-5.143038,2.703305e-07,1.534306e-04,None,Deseq2,True
1,POGZ,27978.631933,-1.958426,0.071577,-27.361215,7.942751e-165,6.762061e-161,None,Deseq2,True
710,CFAP95,396.549814,-1.946532,1.125921,-1.728836,8.383851e-02,9.994596e-01,None,Deseq2,False
0,LETMD1,6347.383442,-1.887973,0.062211,-30.348026,2.667753e-202,4.542384e-198,None,Deseq2,True
...,...,...,...,...,...,...,...,...,...,...
332,ENSG00000276256,934.394655,2.742878,1.302659,2.105599,3.523918e-02,9.994596e-01,None,Deseq2,False
263,HCG24,1075.958193,3.824784,1.708837,2.238238,2.520554e-02,9.994596e-01,None,Deseq2,False
80,ENSG00000236209,924.938249,3.989158,1.244702,3.204909,1.351051e-03,2.817395e-01,None,Deseq2,False
31,PKDCC,327.908227,4.395666,0.893746,4.918250,8.732120e-07,4.646307e-04,None,Deseq2,True


# Plot Results

In [5]:
df_true_degs = pd.read_csv(TRUE_DEG_PATH)
df_true_degs


,gene_id,log2fc,frac_applied,n_subjects_applied,subjects_applied
0,ENSG00000237280,0.522222,0.3,3,"MD9129,MS986638,UMBEB23076"
1,ENSG00000250027,0.733333,0.7,7,"MD9129,MS30499,MS706984,MS876075,MS986638,UMBE..."
2,ENSG00000289321,-0.733333,0.4,4,"MD6927,MS30499,MS876075,MS986638"
3,OTULIN,0.100000,0.6,6,"MD6927,MD9129,MS30499,MS876075,MS986638,UMBEB2..."
4,MYLIP,-0.522222,0.2,2,"MD6927,UMBEB22074"
...,...,...,...,...,...
395,FLVCR2,0.733333,0.1,1,UMBEB23158
396,PTPN20,1.577778,0.5,5,"MD6927,MS30499,MS706984,UMBEB22074,UMBEB23158"
397,RACGAP1,-0.100000,0.7,7,"MD6927,MD9129,MS30499,UMBEB22074,UMBEB23076,UM..."
398,ENSG00000257239,2.000000,0.5,5,"MS706984,MS876075,MS986638,UMBEB23076,UMBEB24013"


In [ ]:
for dataset in sorted(os.listdir(DATASET_FOLDER)):

    print(f"\n######\n Processing dataset: {dataset} \n######\n")

    results_folder = os.path.join(DATASET_FOLDER, dataset, METHOD_NAME)
    save_csv_path = os.path.join(results_folder, f"{dataset}-{METHOD_NAME}_results.csv")

    df_nebula = pd.read_csv(save_csv_path)

    DEG.plot_nebula_results(df_nebula, 
                    df_true_degs, 
                    col_p=f"p_{CONTRAST_VARIABLE}{CONTRAST_STIM}", 
                    col_logFC=f"logFC_{CONTRAST_VARIABLE}{CONTRAST_STIM}", 
                    col_gene="gene",
                    PVAL_THR=PVAL_THR,
                    LOGFC_THR=LOGFC_THR,
                    max_FDR=10,
                    add_adj_p_col=True
                    )
    #break